# 추론 코드

# 설치 요소

In [3]:
!pip install nptdms

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.5/181.5 kB 3.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for nptdms: filename=nptdms-1.10.0-py3-none-any.whl size=108456 sha256=7d219f201b4e79cd86125e55278fb4a3b1a82afe7eb26363926d150ccd200f33
  Stored in directory: /root/.cache/pip/wheels/1b/4b/17/21e8b03b37ea51ce7ec9f5570cdf0decca93f537d61c06880f
Successfully built nptdms


# 라이브러리

In [5]:
import zipfile
from nptdms import TdmsFile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import torch
from torch.utils.data import Dataset
from tqdm import tqdm
import math
import zipfile
import os
from nptdms import TdmsFile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import ipywidgets as widgets
from IPython.display import display
import os
import pandas as pd
import torch
from torch.utils.data import Dataset
from tqdm import tqdm
import math
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split


# 테스트 데이터 셋 클래스

In [24]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset
from tqdm import tqdm

class PreloadedTestDatasetFromCSV(Dataset):
    def __init__(self, test_root_dir):
        """
        test_root_dir: "Test_set" 폴더 경로 (하위에 Test1 ~ Test6 폴더 존재)
        각 하위 폴더에 있는 *.csv 파일을 로드하여 test 데이터로 구성
        """
        self.X_list = []
        self.file_list = []

        # 평균-표준편차 (validation 기준)
        torque_mean = -6.08
        torque_std = 1.68
        temp1_mean = 102.80
        temp1_std = 12.30
        temp2_mean = 115
        temp2_std = 16.53

        test_folders = sorted([f for f in os.listdir(test_root_dir) if os.path.isdir(os.path.join(test_root_dir, f))])

        for folder in tqdm(test_folders, desc="📁 Test 폴더 탐색"):
            folder_path = os.path.join(test_root_dir, folder)
            csv_files = sorted([f for f in os.listdir(folder_path) if f.endswith(".csv")])

            for fname in csv_files:
                file_path = os.path.join(folder_path, fname)

                # ⏬ CSV 불러오기
                df = pd.read_csv(file_path)
                # "Time (s)" 열 삭제
                if "Time (s)" in df.columns:
                    df.drop(columns=["Time (s)"], inplace=True)

                # ⚙️ 평균-표준편차 기반 정규화
                if 'Torque[Nm]' in df.columns:
                    df['Torque[Nm]'] = (df['Torque[Nm]'] - torque_mean) / torque_std
                if 'TC SP Front[℃]' in df.columns:
                    df['TC SP Front[℃]'] = (df['TC SP Front[℃]'] - temp1_mean) / temp1_std
                if 'TC SP Rear[℃]' in df.columns:
                    df['TC SP Rear[℃]'] = (df['TC SP Rear[℃]'] - temp2_mean) / temp2_std

                # 🔁 Tensor 변환
                x_tensor = torch.tensor(df.values.astype('float32')).contiguous()
                self.X_list.append(x_tensor)
                self.file_list.append(fname)  # 또는 f"{folder}/{fname}" 로 전체 경로 추적 가능

    def __len__(self):
        return len(self.X_list)

    def __getitem__(self, idx):
        return self.X_list[idx], self.file_list[idx]


## 모델 정의

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNNLSTM(nn.Module):
    def __init__(self, input_channels=15):
        super(CNNLSTM, self).__init__()

        # Conv1D: 입력 시계열에서 지역적인 특징 추출 (예: 진동의 작은 급변 패턴 감지)
        self.conv1 = nn.Conv1d(in_channels=input_channels, out_channels=64, kernel_size=5)

        # MaxPooling: 시계열 길이 줄이면서 중요한 특징만 남김 (계산 효율 향상)
        self.pool = nn.MaxPool1d(kernel_size=2)

        # LSTM: 지역 특징 시퀀스를 받아서 시간적 패턴(추세, 변화 등)을 학습
        self.lstm = nn.LSTM(input_size=64, hidden_size=64, batch_first=True)

        # FC Layer: 마지막 LSTM 출력을 받아서 RUL 회귀값으로 변환
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)  # RUL 회귀
        )

    def forward(self, x):
        # x: (batch, time_steps, channels)
        x = x.permute(0, 2, 1).contiguous()        # → (batch, channels, time_steps) for Conv1D
        x = F.relu(self.conv1(x))    # Conv + ReLU
        x = self.pool(x)             # Pooling
        x = x.permute(0, 2, 1).contiguous()       # → (batch, time, features) for LSTM
        x, _ = self.lstm(x)          # LSTM sequence output
        x = x[:, -1, :]              # 마지막 시점의 hidden state
        return self.fc(x)            # FC 회귀 결과 (RUL)


In [15]:
model = CNNLSTM(input_channels=15) # 모델 인자 설정
model.load_state_dict(torch.load("./model_weights/best_model_0.66.pth"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

CNNLSTM(
  (conv1): Conv1d(15, 64, kernel_size=(5,), stride=(1,))
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (lstm): LSTM(64, 64, batch_first=True)
  (fc): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
)

## 마지막 시퀀스에 대한 CNN-LSTM 예측

In [26]:
test_dataset = PreloadedTestDatasetFromCSV(test_root_dir="./data/Validation")
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

📁 Test 폴더 탐색: 100%|██████████| 3/3 [00:02<00:00,  1.21it/s]


In [27]:
# 예측값 저장 리스트
predicted_rul_list = []

with torch.no_grad():
    for x_test, fname in test_loader:
        x_test = x_test.to(device)  # 중요!!
        pred = model(x_test)
        rul_sec = torch.expm1(pred).cpu().numpy().flatten()
        predicted_rul_list.extend(rul_sec)

        print(f"{fname[0]} → 예측 RUL (초): {rul_sec[0]:.2f}")

2_modified_KIMM Simulator_KIMM Bearing Test_20160422055414.csv → 예측 RUL (초): 51730.23
1_modified_KIMM Simulator_KIMM Bearing Test_20160321050739.csv → 예측 RUL (초): 30379.84


In [28]:
predicted_rul_list

[np.float32(51730.23), np.float32(30379.844)]

In [29]:
# 제출 파일 불러오기
submit = pd.read_excel("./data/submit/팀이름_validation.xlsx")

# 순서가 맞다고 가정하고 결과 삽입
submit["RUL_Score (sec)"] = predicted_rul_list

FileNotFoundError: [Errno 2] No such file or directory: '팀이름_validation.xlsx'

In [ ]:
submit.to_excel("./data/submit/KIST 베어링 신의 제자들_validation.xlsx", index=False)